In [28]:
import altair as alt
import pandas as pd
import numpy as np
import sqlite3 as sql
import random

from sqlalchemy import create_engine
import sqlalchemy 
from vega_datasets import data

In [29]:
state_codes = pd.read_csv("data/StateCodes.csv", header=0, names=['Name', 'Code', 'Abbreviation'], skipinitialspace=True)

In [30]:
engine = create_engine('sqlite:///BabyNamesFromSocialSecurityCardApplications.db', echo=False)
try:
    with sql.connect("BabyNamesFromSocialSecurityCardApplications.db") as con:
        cur = con.cursor()
        res = cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='BabyNames';")
        
        if res.fetchone() is None:
            for state in state_codes['Abbreviation']:
                df = pd.read_csv(f"data/{state}.txt", header=None, names=["State", "Sex", "Year", "Name", "Count"])
                df.to_sql(name='BabyNames', con=engine, if_exists='append')
                
            state_codes = pd.read_csv("data/StateCodes.csv", header=0, names=['Name', 'Code', 'Abbreviation'], skipinitialspace=True)
            state_codes.to_sql(name='StateCodes', con=engine, if_exists='replace')
            
        
except sql.OperationalError as e:
    print("Failed to open database:", e)

In [50]:
names = None
TopNamesEver = sqlalchemy.text("""
    SELECT Name FROM BabyNames
        GROUP BY Name
        ORDER BY sum(Count) DESC limit 50
    """)

AllNames = sqlalchemy.text("SELECT DISTINCT name FROM BabyNames")

with engine.begin() as con:
    names = pd.read_sql(TopNamesEver, con)

names.sort_values(by='Name', ascending=True, inplace=True)
    
print(names.head())
print(names.count())

       Name
24   Andrew
16  Anthony
19  Barbara
40    Betty
30    Brian
Name    50
dtype: int64


In [51]:
alt.data_transformers.enable('default', max_rows=None)
states = alt.topo_feature(data.us_10m.url, feature='states')

NameData = None
with engine.begin() as con:
    query = sqlalchemy.text("""
        SELECT C.Code, N.Year, N.Name, N.Count, cast(N.Count as REAL) / P.Total * 100 AS "Ratio"
            FROM BabyNames N, StateCodes C, (
                SELECT State, Year, sum(Count) AS "Total" from BabyNames
                    GROUP BY State, Year
            ) AS P
                WHERE P.State = N.State AND P.Year = N.Year
                    AND N.State = C.Abbreviation
                    AND N.Name in (
                        SELECT Name FROM BabyNames
                            GROUP BY Name
                            ORDER BY sum(Count) DESC limit 50
                    )
    """)
    NameData = pd.read_sql(query, con)

In [52]:
slider = alt.binding_range(min=1910, max=2024, step=1, name='Year: ')
year = alt.param(name='Year', value=2010, bind=slider)

search = alt.binding(input='text', placeholder='Search Name...', name='View Name: ')
name_dropdown = alt.binding_select(options=names['Name'].to_list(), name='Selected Name: ')
name_select = alt.param(name='Name', value=names['Name'].to_list()[0], bind=name_dropdown)

frequency = alt.Chart(NameData).mark_geoshape().encode(
        color=alt.condition("datum.Count > 0", alt.Color('Ratio:Q', scale=alt.Scale(domain=(0,2)), legend=alt.Legend(title="% of State's New Births")), alt.value('lightGrey')),
        tooltip=['Name', 'Count', 'Year', 'Ratio']
    ).transform_filter(
        'datum.Year == Year && lower(datum.Name) == lower(Name)'
#         name_select & year
    ).transform_lookup(
        lookup='Code',
        from_=alt.LookupData(states, 'id', ['type', 'properties', 'geometry'])
    ).properties(
        width=1700,
        height=800
    ).project(
        type='albersUsa'
    ).add_params(
        name_select,
        year
    )

background = alt.Chart(states).mark_geoshape(
    fill='lightgray',
    stroke='white',
    ).properties(
    width=1700,
    height=800
    ).project('albersUsa')

chart = (background + frequency)
chart.save("Names.html")